In [ ]:
# Configuración COSMIC v0.0.1 - Estructura Modular
import sys
from pathlib import Path

# Configuración automática de rutas relativas
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parents[2]  # Tres niveles arriba desde data/test/NGC6383/

# Agregar al path si no está
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"🌟 COSMIC v0.0.1 - NGC6383 Preprocessing")
print(f"📁 Directorio actual: {CURRENT_DIR}")
print(f"📁 Proyecto: {PROJECT_ROOT}")

# Verificar instalación de COSMIC
try:
    import cosmic
    print("✅ COSMIC modular disponible")
except ImportError:
    print("⚠️  Instalando COSMIC...")
    import os
    os.system(f"pip install -e {PROJECT_ROOT}")
    import cosmic
    print("✅ COSMIC instalado")

In [ ]:
# === Configuración con nueva estructura modular ===
import os
import glob
import optuna
import dill

# NUEVOS IMPORTS MODULARES (recomendado)
from cosmic.preprocess.preprocessor import DataPreprocessor
from cosmic.io.loader import DataLoader
from cosmic.core.clustering import Clustering

print("📦 Módulos importados:")
print(f"   🔧 DataPreprocessor: {DataPreprocessor}")
print(f"   🔧 DataLoader: {DataLoader}")
print(f"   🔧 Clustering: {Clustering}")

# Configuración del preprocesamiento
BASE_DIR = "data"                 # carpeta que contiene 40, 50, 60, 70 ...
PROJECT_PREFIX = "NGC_6383"       # prefijo de los ECSV (ajústalo si cambia)
force = True                      # True para recalcular aunque exista clustering_results.ecsv
use_last = False

OPTUNA_SPACE_DEFAULT = {
    'min_cluster_size': {'type': 'int', 'low': 10, 'high': 200, 'log': False},
    'min_samples':      {'type': 'int', 'low': 10, 'high': 200, 'log': False},
}
HDBSCAN_KW_DEFAULT = {
    "cluster_selection_method": "eom",
    "allow_single_cluster": False,
    "max_cluster_size": 1000,
}

# ── Helpers ──────────────────────────────────────────────────────────────────

def find_ecsv(folder: str) -> str | None:
    """Devuelve el primer archivo *-result.ecsv encontrado en folder, o None."""
    matches = glob.glob(os.path.join(folder, "*-result.ecsv"))
    return matches[0] if matches else None


def get_last_study_name(sqlite_path: str) -> str | None:
    """Retorna el nombre del estudio Optuna más reciente en la DB dada."""
    storage_url = f"sqlite:///{sqlite_path}"
    summaries = optuna.get_all_study_summaries(storage=storage_url)
    if not summaries:
        return None
    return sorted(summaries, key=lambda s: s.datetime_start)[-1].study_name

In [ ]:
# === 2) Descubrir subcarpetas numéricas (40, 50, 60, 70 ...) y mostrarlas ===
subdirs = sorted(
    d for d in os.listdir(BASE_DIR)
    if os.path.isdir(os.path.join(BASE_DIR, d)) and d.isdigit()
)
print("Subcarpetas encontradas:", subdirs)

In [ ]:
for d in subdirs:
    folder = os.path.join(BASE_DIR, d)
    print(f"\n=== Carpeta: {folder} ===")

    ecsv = find_ecsv(folder)
    if not ecsv:
        print("  → No encontré '*-result.ecsv'. Siguiente.")
        continue

    out_ecsv    = os.path.join(folder, "clustering_results.ecsv")
    out_dill    = os.path.join(folder, "clustering_results.dill")
    sqlite_path = os.path.join(folder, "optuna_study.db")

    if (not force) and os.path.exists(out_ecsv):
        print(f"  → Ya existe {os.path.basename(out_ecsv)}. (force=False) Siguiente.")
        continue

    # ---- Carga y preprocesado ----
    print(f"  → Leyendo: {os.path.basename(ecsv)}")
    loader = DataLoader(ecsv)
    data = loader.load_data(
        systems=['Gaia','TMASS','WISE'],
        include_distances=['geometric'],
        include_zp_cols=True,
        include_flux_errors=True,
        fidelity='fidelity_v2'
    )

    pre = DataPreprocessor(data)
    pre.rename_columns()
    pre.drop_invalid_sources()
    pre.fill_missing_values()
    pre.apply_zero_point_correction()
    pre.correct_proper_motion()
    pre.add_photometric_errors()
    good_data, bad_data = pre.filter_data(fidelity_threshold=0.5)

    # ---- Clustering ----
    db_exists = os.path.exists(sqlite_path)
    last_name = get_last_study_name(sqlite_path) if db_exists else None

    if use_last and db_exists and last_name:
        # Reutiliza el mejor estudio ya optimizado: solo re-fit con sus best_params
        print(f"  → Reutilizando estudio Optuna: {last_name}")
        clust = Clustering(good_data, bad_data,
                           search_method='optuna',
                           sqlite_path=sqlite_path,
                           study_name=last_name)
        clust.search(
            columns=["pmra", "pmdec"],
            optuna_search_space={},
            n_trials=0,               # solo carga; no lanza nuevos trials
            n_jobs=1,                 # irrelevante con 0 trials
            sampler="TPESampler",
            sampler_kwargs={"multivariate": True},
            hdbscan_kwargs=HDBSCAN_KW_DEFAULT,
        )
    else:
        # Crea un estudio nuevo (con o sin DB previa, con o sin use_last)
        if use_last and not (db_exists and last_name):
            print("  → use_last=True pero no hay estudio previo. Creando uno nuevo.")
        elif db_exists:
            print("  → DB presente, use_last=False → nuevo estudio en la misma DB.")
        else:
            print("  → Sin DB previa. Creando DB y nuevo estudio.")

        clust = Clustering(good_data, bad_data,
                           search_method='optuna',
                           sqlite_path=sqlite_path)
        clust.search(
            columns=["pmra", "pmdec"],
            optuna_search_space=OPTUNA_SPACE_DEFAULT,
            n_trials=600,
            n_jobs=-1,
            sampler="TPESampler",
            sampler_kwargs={"multivariate": True},
            hdbscan_kwargs=HDBSCAN_KW_DEFAULT,
        )

    print("Best params:", clust.best_params_)
    print("Best score :", clust.best_score_)
    clust.clustering_statistics()
    clust.get_cluster_summary(include_noise=True)
    clust.plot_cluster_members(show_outliers=False)
    clust.plot_cluster_persistence()
    clust.plot_pm_scatter(show_outliers=False)
    # ---- Guardar ----
    clust.save_results(out_ecsv, format='ascii.ecsv')
    with open(out_dill, "wb") as f:
        dill.dump(clust, f, protocol=dill.HIGHEST_PROTOCOL)
    print(f"  → Guardado: {os.path.basename(out_ecsv)}, {os.path.basename(out_dill)}")